# **Initialization**

In [1]:
print('Start')

Start


In [13]:
%load_ext autoreload
#%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import pulp
import vrplib
import re
import sys
import os
import gc
import glob
import contextlib
import modified_didppy as m_dp
import time as pytime

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **Data**

In [14]:
def read_formated_data(file_path):
    """
    Reads a Solomon format .txt file using vrplib and returns a dictionary 
    formatted for CVRPTW LP Relaxation and DIDP models.
    
    Ensures all numerical data (demand, time windows, service times, costs, capacity) 
    are returned as floats.
    """
    # 1. Read instance using vrplib
    # instance_format='solomon' ensures correct parsing of sections
    instance = vrplib.read_instance(file_path, instance_format='solomon')

    # 2. Extract Data & Cast to Float
    # 'edge_weight' is the distance matrix computed by vrplib
    travel_cost = instance['edge_weight'].astype(float).tolist()
    
    # 'node_coord' is available if you ever need it, but we use the pre-calc weights
    num_locations = len(instance['node_coord'])

    # 3. Return Bundle
    return {
        'num_locations': num_locations,
        'num_vehicles': int(instance.get('vehicles', 25)), 
        'capacity': float(instance['capacity']),
        
        # Cast demand to float list
        'demand': instance['demand'].astype(float).tolist(),
        
        # Cast Time Windows to float list
        # Col 0 is ready_time (earliest arrival), Col 1 is due_date (latest arrival)
        'ready_time': instance['time_window'][:, 0].astype(float).tolist(),
        'due_date': instance['time_window'][:, 1].astype(float).tolist(),
        
        # Cast Service Time to float list
        'service_time': instance['service_time'].astype(float).tolist(),
        
        # Use the pre-computed edge weights from vrplib
        'travel_cost': travel_cost
    }
    
def get_best_known_solution(instance_file_path, bk_dict=None):
    """
    1. Tries to find the cost in the provided dictionary (bk_dict).
       - Normalizes filename to lowercase and removes extension to match dict keys.
    2. If not found or no dict provided, looks for a corresponding .sol file.
    """
    
    # --- 1. Dictionary Lookup ---
    if bk_dict:
        # Extract filename (e.g., "C1_2_1.TXT")
        filename = os.path.basename(instance_file_path)
        # Remove extension and convert to lowercase (e.g., "c1_2_1")
        key_name = os.path.splitext(filename)[0].lower()
        
        if key_name in bk_dict:
            return bk_dict[key_name]

    # --- 2. Fallback: .sol file lookup ---
    base_path = instance_file_path.rsplit('.', 1)[0]
    sol_path = base_path + '.sol'
    
    if os.path.exists(sol_path):
        try:
            solution = vrplib.read_solution(sol_path)
            return solution.get('cost', None) 
        except Exception as e:
            # print(f"Warning: Could not read solution file {sol_path}: {e}")
            return None
            
    return None

In [15]:
# These variable names match what is typically expected by your DIDP/LP models
current_num_locations = 2
current_num_vehicles  = 2
current_capacity      = 10.0
current_cust_demand   = [0.0, 0.0]
current_avail_time    = [0.0, 0.0]
current_due_date      = [1000.0, 1000.0]
current_serve_time  = [0.0, 0.0]
current_travel_cost   = [[[0.0] for i in range(1,3)] for row in range(1,3)]

# Directory containing the VRP instances (Update this path)
# Note: Ensure this folder contains your .vrp files (e.g., A-n33-k5.vrp)
Solomon_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\Solomon"
HG_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\homberger_200_customer_instances"
folder_path = HG_folder_path
#folder_path = Solomon_folder_path
# Get all .vrp files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select random instances (or all)
num_instances_to_test = 1000
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)

Found 60 files. Selected 60 for testing.
Selected Instances:
 - C1_2_1.txt
 - C1_2_10.TXT
 - C1_2_2.TXT
 - C1_2_3.TXT
 - C1_2_4.TXT
 - C1_2_5.TXT
 - C1_2_6.TXT
 - C1_2_7.TXT
 - C1_2_8.TXT
 - C1_2_9.TXT
 - C2_2_1.TXT
 - C2_2_10.TXT
 - C2_2_2.TXT
 - C2_2_3.TXT
 - C2_2_4.TXT
 - C2_2_5.TXT
 - C2_2_6.TXT
 - C2_2_7.TXT
 - C2_2_8.TXT
 - C2_2_9.TXT
 - R1_2_1.TXT
 - R1_2_10.TXT
 - R1_2_2.TXT
 - R1_2_3.TXT
 - R1_2_4.TXT
 - R1_2_5.TXT
 - R1_2_6.TXT
 - R1_2_7.TXT
 - R1_2_8.TXT
 - R1_2_9.TXT
 - R2_2_1.TXT
 - R2_2_10.TXT
 - R2_2_2.TXT
 - R2_2_3.TXT
 - R2_2_4.TXT
 - R2_2_5.TXT
 - R2_2_6.TXT
 - R2_2_7.TXT
 - R2_2_8.TXT
 - R2_2_9.TXT
 - RC1_2_1.TXT
 - RC1_2_10.TXT
 - RC1_2_2.TXT
 - RC1_2_3.TXT
 - RC1_2_4.TXT
 - RC1_2_5.TXT
 - RC1_2_6.TXT
 - RC1_2_7.TXT
 - RC1_2_8.TXT
 - RC1_2_9.TXT
 - RC2_2_1.TXT
 - RC2_2_10.TXT
 - RC2_2_2.TXT
 - RC2_2_3.TXT
 - RC2_2_4.TXT
 - RC2_2_5.TXT
 - RC2_2_6.TXT
 - RC2_2_7.TXT
 - RC2_2_8.TXT
 - RC2_2_9.TXT
--------------------------------------------------


# **1. Model and dual bound declaration**

In [16]:
def creation_of_didp_model_function():
    num_locations = current_num_locations
    num_vehicles = current_num_vehicles
    q = current_capacity
    cust_demand = current_cust_demand
    avail_time = current_avail_time
    due_date = current_due_date
    serve_time = current_serve_time
    travel_cost = current_travel_cost
    
    # =====================================================================================
    # DIDP Model Definition
    # =====================================================================================
    model = m_dp.Model(float_cost=True)

    # Object types for customers/locations and vehicles
    customer = model.add_object_type(number=num_locations)
    vehicle = model.add_object_type(number=num_vehicles)

    # -------------------- State Variables --------------------
    # Set of unvisited customers
    unvisited_locations = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))

    # Per-vehicle state variables, stored in Python lists for easy access
    vehicle_locations = [
    model.add_element_var(object_type=customer, target=0, name=f"loc_v{v}")
    for v in range(num_vehicles)
    ]
    vehicle_loads = [
    model.add_float_var(target=0, name=f"load_v{v}")
    for v in range(num_vehicles)
    ]
    vehicle_times = [
    model.add_float_resource_var(target=0, less_is_better=True, name=f"time_v{v}")
    for v in range(num_vehicles)
    ]
    chosen_customer = model.add_element_var(object_type=customer, target=0, name="chosen_customer")
    alpha = model.add_int_var(target=0, name="alpha")

    # -------------------- Tables of Constants --------------------
    demand = model.add_float_table(cust_demand)
    ready_time = model.add_float_table(avail_time)
    due_time = model.add_float_table(due_date)
    service_time = model.add_float_table(serve_time)
    travel_time = model.add_float_table(travel_cost)

    # -------------------- Transitions --------------------
    # Choose customer j to be visited next
    for j in range(1, num_locations):
        choosing_customer_transition = m_dp.Transition(
            name=f"choose_customer_{j}_to_visit",
            cost=m_dp.FloatExpr.state_cost(),
            preconditions =[
                unvisited_locations.contains(j),
                alpha == 0
                ],
            effects=[
                (chosen_customer, j),
                (alpha, 1)
                ],
        )
        model.add_transition(choosing_customer_transition)

        # Transition to visit a customer j with a vehicle v
    for v in range(num_vehicles):
        arrival_time = m_dp.max(
            vehicle_times[v] + travel_time[vehicle_locations[v], chosen_customer],
            ready_time[chosen_customer]
        )

        departure_time = arrival_time + service_time[chosen_customer]

        visit_transition = m_dp.Transition(
            name=f"visit_chosen_customer_with_vehicle_{v}",
            cost=travel_time[vehicle_locations[v], chosen_customer] + m_dp.FloatExpr.state_cost(),
            preconditions=[
                unvisited_locations.contains(chosen_customer),
                vehicle_loads[v] + demand[chosen_customer] <= q,
                arrival_time <= due_time[chosen_customer],
                alpha == 1,
            ],
            effects=[
                (unvisited_locations, unvisited_locations.remove(chosen_customer)),
                (vehicle_locations[v], chosen_customer),
                (vehicle_loads[v], vehicle_loads[v] + demand[chosen_customer]),
                (vehicle_times[v], departure_time),
                (alpha, 0),
            ],
        )

        model.add_transition(visit_transition)

    # Transitions for each vehicle to return to the depot after all customers are served
    for v in range(num_vehicles):
        return_to_depot_transition = m_dp.Transition(
            name=f"return_vehicle_{v}_to_depot",
            cost=travel_time[vehicle_locations[v], 0] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited_locations.is_empty(), vehicle_locations[v] != 0],
            effects=[(vehicle_locations[v], 0)],
        )
        model.add_transition(return_to_depot_transition)

    # --- 1. Global Capacity Constraint (The Efficient "Cut") ---
    # Logic: Total Capacity Available >= Total Demand Remaining
    # Summing variables in Python creates a DIDP expression automatically
    total_current_load = sum(vehicle_loads) 
    total_fleet_capacity = num_vehicles * q
    
    # demand[unvisited_locations] automatically sums the weight of items in the set
    model.add_state_constr(
        (total_fleet_capacity - total_current_load) >= demand[unvisited_locations]
    )
    
    # -------------------- Base Case --------------------
    # All customers visited AND all vehicles are at the depot
    base_conditions = [unvisited_locations.is_empty()]
    for v in range(num_vehicles):
        base_conditions.append(vehicle_locations[v] == 0)
    model.add_base_case(base_conditions)

    # =========================================================
    # Dual Bounds
    # =========================================================

    # --- Pre-computation of Min Edge Tables ---
    # min_from[i]: The cheapest cost to LEAVE node i
    min_from_val = [min(travel_cost[i][k] for k in range(num_locations) if k != i) for i in range(num_locations)]
    min_from = model.add_float_table(min_from_val)

    # min_to[j]: The cheapest cost to ENTER node j
    min_to_val = [min(travel_cost[k][j] for k in range(num_locations) if k != j) for j in range(num_locations)]
    min_to = model.add_float_table(min_to_val)

    # --- Dual Bound 1: Minimum Outgoing Edges ---
    # Logic: 
    # 1. We must leave every customer that is currently unvisited.
    # 2. Every vehicle that is currently NOT at the depot must leave its current location.
    lb_outgoing = min_from[unvisited_locations]  # Sum of min_from for all unvisited nodes
    for v in range(num_vehicles):
        # If vehicle v is at a customer (location != 0), it must leave that customer eventually.
        # We add the min cost to leave its current location.
        lb_outgoing += (vehicle_locations[v] != 0).if_then_else(min_from[vehicle_locations[v]], 0.0)
    model.add_dual_bound(lb_outgoing)
    
    # --- Dual Bound 2: Minimum Incoming Edges ---
    # Logic:
    # 1. We must enter every customer that is currently unvisited.
    # 2. Every vehicle that is currently NOT at the depot must eventually return (enter) the depot.
    lb_incoming = min_to[unvisited_locations] # Sum of min_to for all unvisited nodes
    for v in range(num_vehicles):
        # If vehicle v is out working (location != 0), it must return to depot (enter node 0).
        lb_incoming += (vehicle_locations[v] != 0).if_then_else(min_to[0], 0.0)
    model.add_dual_bound(lb_incoming)

    # =========================================================
    # 3. Bundle Metadata
    # =========================================================
    metadata = {
        "unvisited_locations": unvisited_locations,
        "vehicle_locations": vehicle_locations,
        "vehicle_loads": vehicle_loads,
        "vehicle_times": vehicle_times,
        "distance_matrix": travel_cost,
        "demand": cust_demand,
        "due_time": due_date,
        "ready_time": avail_time,
        "service_time": serve_time,
        "capacity": q,
        "num_vehicles": num_vehicles,
        "num_locations": num_locations
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# **Execution**

In [17]:
results_data = []
output_csv_name = "CVRPTW_single_dual_bound_HG_results_10s_lim.csv"

best_known_costs_HG_200 = {
    # 200 Customers
    "c1_2_1": 2704.57, "c1_2_2": 2917.89, "c1_2_3": 2707.35, "c1_2_4": 2643.31, "c1_2_5": 2702.05, 
    "c1_2_6": 2701.04, "c1_2_7": 2701.04, "c1_2_8": 2775.48, "c1_2_9": 2687.83, "c1_2_10": 2643.51,
    "c2_2_1": 1931.44, "c2_2_2": 1863.16, "c2_2_3": 1775.08, "c2_2_4": 1703.43, "c2_2_5": 1878.85, 
    "c2_2_6": 1857.35, "c2_2_7": 1849.46, "c2_2_8": 1820.53, "c2_2_9": 1830.05, "c2_2_10": 1806.58,
    "r1_2_1": 4784.11, "r1_2_2": 4039.86, "r1_2_3": 3381.96, "r1_2_4": 3057.81, "r1_2_5": 4107.86, 
    "r1_2_6": 3583.14, "r1_2_7": 3150.11, "r1_2_8": 2951.99, "r1_2_9": 3760.58, "r1_2_10": 3301.18,
    "r2_2_1": 4483.16, "r2_2_2": 3621.20, "r2_2_3": 2880.62, "r2_2_4": 1981.29, "r2_2_5": 3366.79, 
    "r2_2_6": 2913.03, "r2_2_7": 2451.14, "r2_2_8": 1849.87, "r2_2_9": 3092.04, "r2_2_10": 2654.97,
    "rc1_2_1": 3602.80, "rc1_2_2": 3249.05, "rc1_2_3": 3008.33, "rc1_2_4": 2851.68, "rc1_2_5": 3371.00, 
    "rc1_2_6": 3324.80, "rc1_2_7": 3189.32, "rc1_2_8": 3083.93, "rc1_2_9": 3081.13, "rc1_2_10": 3000.30,
    "rc2_2_1": 3099.53, "rc2_2_2": 2825.24, "rc2_2_3": 2601.87, "rc2_2_4": 2038.56, "rc2_2_5": 2911.46, 
    "rc2_2_6": 2873.12, "rc2_2_7": 2525.83, "rc2_2_8": 2292.53, "rc2_2_9": 2175.04, "rc2_2_10": 2015.60
    }

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    # --- NEW: Get Best Known Solution ---
    best_known_cost = get_best_known_solution(file_path, bk_dict=best_known_costs_HG_200)
    
    try:
        # --- A. Read Data & Update Globals ---
        data = read_formated_data(file_path)

        # Extract variables for global usage (matching your model function)
        current_num_locations = data['num_locations']
        current_num_vehicles  = data['num_vehicles']
        current_capacity      = data['capacity']
        current_cust_demand   = data['demand']
        current_avail_time    = data['ready_time']
        current_due_date      = data['due_date']
        current_serve_time    = data['service_time']
        current_travel_cost   = data['travel_cost']
        
        # --- B. Initialize Model ---
        model, state_data = creation_of_didp_model_function()
        
        # --- C. Solver Execution ---
        t_start = pytime.time()
        
        # Standard CABS solver with 30-minute limit
        solver = m_dp.CABS(
            model,
            quiet=False,
            time_limit=10
        )
        
        solution = solver.search()
        
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- D. Logging Results ---
        if solution.is_optimal:
            cost = solution.cost
            status = "True"
        elif solution.cost is not None:
            cost = solution.cost
            status = "False (Time Limit)"
        else:
            cost = float('inf')
            status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        # --- Calculate Gap ---
        gap = "N/A"
        if best_known_cost is not None and cost != float('inf') and cost != "Inf":
            try:
                gap_val = ((cost - best_known_cost) / best_known_cost) * 100
                gap = f"{gap_val:.2f}%"
            except:
                gap = "Error"

        # Print Summary
        print(f"Number of locations: {current_num_locations}")
        print(f"Number of vehicles: {current_num_vehicles}")
        print(f"Total Capacity: {current_capacity*current_num_vehicles}")
        print(f"Total Demand: {sum(current_cust_demand)}")
        print(f"   -> My Cost: {cost} | Best Known: {best_known_cost} | Gap: {gap}")
        print(f"   -> Time: {duration:.2f}s | Optimal: {status}")

        results_data.append({
            "Instance": instance_name,
            "Best Known Cost": best_known_cost, # <--- NEW
            "Cost": cost,
            "Gap to BKS": gap,                      # <--- NEW
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status,
            "Infeasibility": solution.is_infeasible
        })

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        # import traceback; traceback.print_exc()
        
        results_data.append({
            "Instance": instance_name,
            "Best Known Cost": best_known_cost,
            "Cost": "Error",
            "Gap to BKS": "N/A",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error",
            "Infeasibility": "Error"
        })

    # --- E. Intermediate Save ---
    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_name, index=False)

print("\n" + "="*50)
print("Batch Testing Complete.")
print(f"Results saved to {output_csv_name}")
print(df_results[['Instance', 'Best Known Cost', 'Cost', 'Gap to BKS', 'Running Time (s)']])


[1/60] Processing: C1_2_1.txt
Number of locations: 201
Number of vehicles: 50
Total Capacity: 10000.0
Total Demand: 3530.0
   -> My Cost: inf | Best Known: 2704.57 | Gap: N/A
   -> Time: 10.02s | Optimal: False (No Sol)

[2/60] Processing: C1_2_10.TXT
Number of locations: 201
Number of vehicles: 50
Total Capacity: 10000.0
Total Demand: 3530.0
   -> My Cost: 6667.414496086404 | Best Known: 2643.51 | Gap: 152.22%
   -> Time: 10.09s | Optimal: False (Time Limit)

[3/60] Processing: C1_2_2.TXT
Number of locations: 201
Number of vehicles: 50
Total Capacity: 10000.0
Total Demand: 3530.0
   -> My Cost: inf | Best Known: 2917.89 | Gap: N/A
   -> Time: 10.01s | Optimal: False (No Sol)

[4/60] Processing: C1_2_3.TXT
Number of locations: 201
Number of vehicles: 50
Total Capacity: 10000.0
Total Demand: 3530.0
   -> My Cost: 7306.722174001556 | Best Known: 2707.35 | Gap: 169.88%
   -> Time: 10.08s | Optimal: False (Time Limit)

[5/60] Processing: C1_2_4.TXT
Number of locations: 201
Number of vehic

KeyError: "['Best Known Cost'] not in index"

# **Extraction**

In [20]:
import pandas as pd
import os

# ==========================================
# 1. Solomon Dataset Selection
# ==========================================

# Manually select specific Solomon instances
# Note: Ensure these names match exactly what is in your results CSV "Instance" column
solomon_selected_instances = [
    "C104.txt",
    "C201.txt",
    "R104.txt",
    "R203.txt",
    "RC104.txt",
    #"RC201.txt"
]

# File path to the full Solomon results CSV
solomon_csv_path = "CVRPTW_single_dual_bound_Solomon_results_10s_lim.csv"

# Check if file exists to avoid errors
if os.path.exists(solomon_csv_path):
    print(f"Processing Solomon results from: {solomon_csv_path}")
    
    # Read the full results CSV
    df_solomon = pd.read_csv(solomon_csv_path)

    # Filter for selected instances only
    df_solomon_selected = df_solomon[df_solomon["Instance"].isin(solomon_selected_instances)]

    # Save to a new intermediate CSV
    df_solomon_selected.to_csv("CVRPTW_selected_Solomon_results.csv", index=False)
    print(f" - Selected {len(df_solomon_selected)} Solomon instances.")
else:
    print(f"Warning: File {solomon_csv_path} not found. Creating empty DataFrame for Solomon.")
    df_solomon_selected = pd.DataFrame()


# ==========================================
# 2. H&G Dataset Selection
# ==========================================

# Manually select specific H&G instances
hg_selected_instances = [
    "c1_2_10.txt",
    "c2_2_10.txt",
    "r1_2_10.txt",
    "r2_2_10.txt",
    "rc2_2_10.txt"
]

# File path to the full H&G results CSV (filename from your previous code)
hg_csv_path = "CVRPTW_single_dual_bound_HG_results_10s_lim.csv"

if os.path.exists(hg_csv_path):
    print(f"Processing H&G results from: {hg_csv_path}")

    # Read the full results CSV
    df_hg = pd.read_csv(hg_csv_path)

    # Filter for selected instances only
    # Note: If your CSV has "C1_2_1.TXT" but list has "c1_2_1.txt", this filter might miss.
    # The line below ensures case-insensitive matching if needed, otherwise just use .isin()
    # df_hg_selected = df_hg[df_hg["Instance"].isin(hg_selected_instances)] 
    
    # Robust filtering (case-insensitive check):
    df_hg["Instance_Lower"] = df_hg["Instance"].astype(str).str.lower()
    selected_lower = [x.lower() for x in hg_selected_instances]
    df_hg_selected = df_hg[df_hg["Instance_Lower"].isin(selected_lower)].drop(columns=["Instance_Lower"])

    # Save to a new intermediate CSV
    df_hg_selected.to_csv("CVRPTW_selected_HG_results.csv", index=False)
    print(f" - Selected {len(df_hg_selected)} H&G instances.")
else:
    print(f"Warning: File {hg_csv_path} not found. Creating empty DataFrame for H&G.")
    df_hg_selected = pd.DataFrame()


# ==========================================
# 3. Combine Results
# ==========================================

# Combine Solomon and H&G selected results
df_combined = pd.concat([df_solomon_selected, df_hg_selected], ignore_index=True)

# Define final output filename
final_output_name = "CVRPTW_single_dual_bound_results_10s_lim.csv"

# Save to the new CSV
df_combined.to_csv(final_output_name, index=False)

print("\n" + "="*50)
print(f"Combined CSV created: {final_output_name}")
print(f"Total rows: {len(df_combined)}")
print("="*50)

# Optional: Preview the combined data
print(df_combined)

Processing Solomon results from: CVRPTW_single_dual_bound_Solomon_results_10s_lim.csv
 - Selected 5 Solomon instances.
Processing H&G results from: CVRPTW_single_dual_bound_HG_results_10s_lim.csv
 - Selected 5 H&G instances.

Combined CSV created: CVRPTW_single_dual_bound_results_10s_lim.csv
Total rows: 10
       Instance  Best Known Cost         Cost Gap to BKS  Nodes Expanded  \
0      C104.txt           822.90  1397.802676     69.86%           10783   
1      C201.txt           589.10  1483.478168    151.82%           13605   
2      R104.txt           971.50  1717.238026     76.76%            8150   
3      R203.txt           870.80  1760.245890    102.14%            7025   
4     RC104.txt          1132.30  1776.870207     56.93%            6869   
5   C1_2_10.TXT          2643.51  6667.414496    152.22%            3151   
6   C2_2_10.TXT          1806.58  5455.235510    201.96%            3257   
7   R1_2_10.TXT          3301.18  7594.405182    130.05%            2950   
8   R2_2

# **Run single dual bound model on selected instances**

In [22]:
# =============================================================================
# 1. SETUP: Directories & Inputs
# =============================================================================

# Paths to your datasets
Solomon_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\Solomon"
HG_folder_path = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\4_CVRPTW_dual_bounds_and_models\Datasets\homberger_200_customer_instances"

# Input CSV containing the instances to run
input_csv_path = "CVRPTW_single_dual_bound_results_10s_lim.csv"

# =============================================================================
# 2. READ INSTANCES & LOCATE FILES
# =============================================================================

print(f"Reading target instances from: {input_csv_path}")
try:
    df_input = pd.read_csv(input_csv_path)
    # Extract the 'Instance' column
    if 'Instance' in df_input.columns:
        all_target_instances = df_input['Instance'].dropna().astype(str).tolist()
    else:
        # Fallback if first column is the instance name but labeled differently
        all_target_instances = df_input.iloc[:, 0].dropna().astype(str).tolist()
    
    print(f"Found {len(all_target_instances)} instances in CSV.")

except Exception as e:
    print(f"Error reading input CSV: {e}")
    all_target_instances = []

# Function to find file in multiple folders (case-insensitive)
def get_full_path(filename, folder_paths):
    for folder in folder_paths:
        for name_variant in [filename, filename.lower(), filename.upper()]:
            full_path = os.path.join(folder, name_variant)
            if os.path.exists(full_path):
                return full_path
    return None

files_to_run = []
folders = [Solomon_folder_path, HG_folder_path]

print("Locating files on disk...")
found_count = 0
for inst_name in all_target_instances:
    path = get_full_path(inst_name, folders)
    if path:
        files_to_run.append(path)
        found_count += 1
    else:
        print(f"Warning: Could not locate file for '{inst_name}' in specified folders.")

print(f"Successfully located {found_count} out of {len(all_target_instances)} files.")

Reading target instances from: CVRPTW_single_dual_bound_results_10s_lim.csv
Found 10 instances in CSV.
Locating files on disk...
Successfully located 10 out of 10 files.


In [23]:
# =============================================================================
# 2. RESUME LOGIC & EXECUTION
# =============================================================================

output_csv_name = "CVRPTW_single_dual_bound_results_1800s_lim.csv"
processed_instances = []

# Check if CSV exists to resume
if os.path.exists(output_csv_name):
    try:
        df_existing = pd.read_csv(output_csv_name)
        if "Instance" in df_existing.columns:
            processed_instances = df_existing["Instance"].tolist()
            print(f"Found existing results file. {len(processed_instances)} instances already processed.")
    except Exception as e:
        print(f"Warning: Could not read existing CSV ({e}). Starting fresh.")

# Filter out files that are already done
# Note: 'files_to_run' is assumed to be defined in your previous cell
remaining_files = [f for f in files_to_run if os.path.basename(f) not in processed_instances]
print(f"Starting execution on {len(remaining_files)} remaining instances (Total selected: {len(files_to_run)}).")

# =============================================================================
# 3. EXECUTION LOOP
# =============================================================================

time_limit_seconds = 1800

for i, file_path in enumerate(remaining_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(remaining_files)}] Processing: {instance_name}")
    
    # --- 1. Get Best Known Cost ---
    best_known_cost = get_best_known_solution(file_path, bk_dict=best_known_costs_HG_200)

    result_entry = {}
    
    try:
        # --- 2. Read Data ---
        data = read_formated_data(file_path)

        # Update globals required for the model function
        current_num_locations = data['num_locations']
        current_num_vehicles  = data['num_vehicles']
        current_capacity      = data['capacity']
        current_cust_demand   = data['demand']
        current_avail_time    = data['ready_time']
        current_due_date      = data['due_date']
        current_serve_time    = data['service_time']
        current_travel_cost   = data['travel_cost']
        
        # --- 3. Create Model ---
        model, state_data = creation_of_didp_model_function()
        
        # --- 4. Run Solver ---
        t_start = pytime.time()
        
        solver = m_dp.CABS(
            model,
            quiet=False,
            time_limit=time_limit_seconds
        )
        
        solution = solver.search()
        t_end = pytime.time()
        duration = t_end - t_start
        
        # --- 5. Parse Results ---
        if solution.is_optimal:
            cost = solution.cost
            status = "True"
        elif solution.cost is not None:
            cost = solution.cost
            status = "False (Time Limit)"
        else:
            cost = float('inf')
            status = "False (No Sol)"

        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        # Gap Calculation
        gap = "N/A"
        if best_known_cost is not None and cost != float('inf') and cost != "Inf":
            try:
                gap_val = ((cost - best_known_cost) / best_known_cost) * 100
                gap = f"{gap_val:.2f}%"
            except:
                gap = "Error"

        print(f"   -> My Cost: {cost} | BKS: {best_known_cost} | Gap: {gap}")
        print(f"   -> Time: {duration:.2f}s | Nodes Exp: {nodes_exp} | Status: {status}")

        result_entry = {
            "Instance": instance_name,
            "Best Known Cost": best_known_cost,
            "Cost": cost,
            "Gap to BKS": gap,
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": status,
            "Infeasibility": solution.is_infeasible
        }

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        result_entry = {
            "Instance": instance_name,
            "Best Known Cost": best_known_cost,
            "Cost": "Error",
            "Gap to BKS": "N/A",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error",
            "Infeasibility": str(e)
        }

    # --- 6. Save Immediately (Append Mode) ---
    df_single_result = pd.DataFrame([result_entry])
    
    # If file doesn't exist, write header. If it exists, append without header.
    if not os.path.exists(output_csv_name):
        df_single_result.to_csv(output_csv_name, index=False)
    else:
        df_single_result.to_csv(output_csv_name, mode='a', header=False, index=False)

print("\n" + "="*50)
print("Batch Execution Complete.")
print(f"Results saved to: {output_csv_name}")

Starting execution on 10 remaining instances (Total selected: 10).

[1/10] Processing: C104.txt
   -> My Cost: 1397.8026756972627 | BKS: 822.9 | Gap: 69.86%
   -> Time: 20.03s | Nodes Exp: 24465 | Status: False (Time Limit)

[2/10] Processing: C201.txt
   -> My Cost: 1483.4781682539556 | BKS: 589.1 | Gap: 151.82%
   -> Time: 20.02s | Nodes Exp: 22778 | Status: False (Time Limit)

[3/10] Processing: R104.txt
   -> My Cost: 1475.9943759256962 | BKS: 971.5 | Gap: 51.93%
   -> Time: 20.01s | Nodes Exp: 14611 | Status: False (Time Limit)

[4/10] Processing: R203.txt
   -> My Cost: 1760.2458897069748 | BKS: 870.8 | Gap: 102.14%
   -> Time: 20.00s | Nodes Exp: 17387 | Status: False (Time Limit)

[5/10] Processing: RC104.txt
   -> My Cost: 1776.870206616999 | BKS: 1132.3 | Gap: 56.93%
   -> Time: 20.01s | Nodes Exp: 17279 | Status: False (Time Limit)

[6/10] Processing: C1_2_10.TXT
   -> My Cost: 6667.414496086404 | BKS: 2643.51 | Gap: 152.22%
   -> Time: 20.04s | Nodes Exp: 5350 | Status: Fal